# Relationship Pipeline End-to-End (Raw Schema)

This notebook is a copy of the main pipeline adapted for the raw production-style schema.

Current business assumptions:
- We use `leader.csv` and `nclq.csv`.
- In `leader.csv`, the working columns are:
  - `cif_khcn`
  - `name_khcn`
  - `cif_khdn`
  - `job_title`
  - `name_khdn` (optional, because the source may omit it)
- In `nclq.csv`, the working columns are:
  - `CIF_NO`
  - `CIF_NO_REL`
  - `Mô tả MQH`
- TH1 `same_person`: keep all non-null job titles.
- TH2 `family`: keep only pairs where both people hold a high role.
- Declared relationships are removed in `hidden_*`:
  - direct `company -> company` rows remove directed company pairs
  - `person -> person` rows remove `family` links at person-pair level


In [ ]:
from pathlib import Path
import re
import time
import unicodedata
import warnings

import pandas as pd

BASE_DIR = Path.cwd()
LEADER_FILE = BASE_DIR / "leader.csv"
LEADER_XLSX_FILE = BASE_DIR / "leader.xlsx"
NCLQ_FILE = BASE_DIR / "nclq.csv"
NCLQ_XLSX_FILE = BASE_DIR / "nclq.xlsx"

OUTPUT_TAG = "raw_schema"

ALL_PAIRS_FILE = BASE_DIR / f"all_relationships_pairs_{OUTPUT_TAG}.csv"
ALL_PEOPLE_FILE = BASE_DIR / f"all_relationships_by_person_{OUTPUT_TAG}.csv"
ALL_XLSX_FILE = BASE_DIR / f"all_relationships_{OUTPUT_TAG}.xlsx"

HIDDEN_PAIRS_FILE = BASE_DIR / f"hidden_relationships_pairs_{OUTPUT_TAG}.csv"
HIDDEN_PEOPLE_FILE = BASE_DIR / f"hidden_relationships_by_person_{OUTPUT_TAG}.csv"
HIDDEN_XLSX_FILE = BASE_DIR / f"hidden_relationships_{OUTPUT_TAG}.xlsx"
DECLARED_DIAGNOSTIC_FILE = BASE_DIR / f"nclq_declared_diagnostics_{OUTPUT_TAG}.csv"

TARGET_ROLE_DIRECTOR = "Giam doc"
TARGET_ROLE_DEPUTY_DIRECTOR = "Pho giam doc"
TARGET_ROLE_LEGAL_REP = "Nguoi dai dien theo phap luat"
TARGET_ROLE_AUTHORIZED = "Nguoi duoc uy quyen cua giam doc/pho giam doc"
TARGET_ROLE_LEADERSHIP = "Ban lanh dao"
TARGET_ROLE_MEMBERS_COUNCIL = "HDTV"
TARGET_ROLE_BOARD_MANAGEMENT = "Ban giam doc"

FAMILY_RELATION_TYPES = {
    "la vo chong cha me con voi ca nhan d",
    "la anh chi em voi ca nhan do",
}
FAMILY_RELATION_CODES = {"201", "211"}
FAMILY_RELATION_CODE_LABELS = {
    "201": "La vo, chong, cha, me, con voi ca nhan d",
    "211": "La anh, chi, em voi ca nhan do",
}

warnings.filterwarnings("ignore", category=FutureWarning)
print("Configured base directory:", BASE_DIR)


In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    if not text:
        return None

    lowered = text.lower()
    if lowered in {"nan", "none", "null", "nat"}:
        return None

    text = text.replace("Đ", "D").replace("đ", "d")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def normalize_header_name(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    text = text.replace("Đ", "D").replace("đ", "d")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9?]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def has_any_phrase(normalized, compact, phrases):
    return any(phrase in normalized or phrase.replace(" ", "") in compact for phrase in phrases)


def classify_high_role(job_title):
    normalized = normalize_text(job_title)
    if not normalized:
        return None

    compact = normalized.replace(" ", "")
    tokens = set(normalized.split())
    is_top_exec = has_any_phrase(
        normalized,
        compact,
        ("tong giam doc", "pho tong giam doc", "tgd", "ptgd", "pho tg"),
    )

    deputy_markers = ("pho giam doc", "pho gd", "p giam doc", "p gd", "pgd", "phogiamdoc", "phogd")
    if has_any_phrase(normalized, compact, deputy_markers) and not is_top_exec:
        return TARGET_ROLE_DEPUTY_DIRECTOR

    legal_rep_markers = (
        "nguoi dai dien theo phap luat",
        "nguoi dai dien phap luat",
        "nguoi dai dien theo pl",
        "nguoi dai dien pl",
        "dai dien theo phap luat",
        "dai dien phap luat",
        "dai dien theo pl",
        "dai dien pl",
    )
    if has_any_phrase(normalized, compact, legal_rep_markers):
        return TARGET_ROLE_LEGAL_REP
    if "dai dien" in normalized and ("phap luat" in normalized or " pl " in f" {normalized} "):
        return TARGET_ROLE_LEGAL_REP

    if has_any_phrase(normalized, compact, ("ban giam doc", "ban gd", "bgd")):
        return TARGET_ROLE_BOARD_MANAGEMENT

    if has_any_phrase(normalized, compact, ("hdtv", "hoi dong thanh vien", "chu tich hdtv", "ct hdtv")):
        return TARGET_ROLE_MEMBERS_COUNCIL

    if has_any_phrase(normalized, compact, ("ban lanh dao", "lanh dao", "lanh dao cty", "lanh dao cong ty", "lanh dao dn")):
        if not has_any_phrase(normalized, compact, ("lanh dao phong", "lanh dao phc")):
            return TARGET_ROLE_LEADERSHIP

    has_authorized_signal = (
        "uy quyen" in normalized
        or bool(tokens & {"uq", "duq", "nduq", "ndduq", "nuq"})
        or any(code in compact for code in ("uqgd", "uqpgd", "nduqgd", "nduqpgd", "duqgd", "duqpgd"))
    )
    if has_authorized_signal:
        excluded_authorized = (
            "ke toan truong",
            "ktt",
            "chu tk",
            "chu tai khoan",
            "chu dn",
            "chu doanh nghiep",
            "tong giam doc",
            "tgd",
            "pho tong giam doc",
            "pho tg",
            "chu tich",
        )
        if not any(term in normalized for term in excluded_authorized):
            director_ref_markers = ("giam doc", "giam oc", "gd", "pho giam doc", "pho gd", "pgd")
            if has_any_phrase(normalized, compact, director_ref_markers):
                return TARGET_ROLE_AUTHORIZED

    if has_any_phrase(normalized, compact, ("giam doc", "giam oc", "gd")) and not is_top_exec:
        return TARGET_ROLE_DIRECTOR

    return None


def unique_list(series):
    seen = set()
    ordered = []
    for value in series:
        if pd.isna(value):
            continue
        if value not in seen:
            seen.add(value)
            ordered.append(value)
    return ordered


def normalize_identifier(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text or None


def make_person_identifier_aligner(existing_person_ids):
    def align(value):
        raw = normalize_identifier(value)
        if raw is None:
            return None
        return raw if raw in existing_person_ids else raw

    return align


def make_company_identifier_aligner(existing_company_ids):
    def align(value):
        raw = normalize_identifier(value)
        if raw is None:
            return None
        return raw if raw in existing_company_ids else raw

    return align


def make_directed_tuple(left, right):
    return (left, right)


def make_undirected_tuple(left, right):
    return tuple(sorted((left, right)))


def resolve_column_names(columns, candidate_map, required_keys):
    normalized_lookup = {}
    for column in columns:
        normalized_lookup[normalize_header_name(column)] = column

    resolved = {}
    for key, candidates in candidate_map.items():
        chosen = None
        for candidate in candidates:
            normalized_candidate = normalize_header_name(candidate)
            if normalized_candidate in normalized_lookup:
                chosen = normalized_lookup[normalized_candidate]
                break
        if chosen is None and key in required_keys:
            raise KeyError(f"Cannot resolve column '{key}'. Available columns: {list(columns)}")
        resolved[key] = chosen
    return resolved


def pick_display_name(name_value, identifier):
    normalized = normalize_text(name_value)
    if normalized is None:
        return identifier
    return str(name_value).strip()


def describe_same_person(row):
    person_id = row["cif_khcn"]
    person_name = pick_display_name(row["name_khcn_A"], person_id)
    role_a = row["job_title_A"]
    role_b = row["job_title_B"]
    company_a = pick_display_name(row["name_khdn_A"], row["cif_khdn_A"])
    company_b = pick_display_name(row["name_khdn_B"], row["cif_khdn_B"])
    return (
        f"Ca nhan '{person_name}' ({person_id}) lam '{role_a}' tai '{company_a}' ({row['cif_khdn_A']}) "
        f"va '{role_b}' tai '{company_b}' ({row['cif_khdn_B']})."
    )


def describe_family(row):
    person_a_id = row["cif_khcn_A"]
    person_b_id = row["cif_khcn_B"]
    person_a_name = pick_display_name(row["name_khcn_A"], person_a_id)
    person_b_name = pick_display_name(row["name_khcn_B"], person_b_id)
    company_a = pick_display_name(row["name_khdn_A"], row["cif_khdn_A"])
    company_b = pick_display_name(row["name_khdn_B"], row["cif_khdn_B"])
    relationship = row["relationship_display"]
    role_a = row["job_title_A"]
    role_b = row["job_title_B"]
    return (
        f"'{person_a_name}' ({person_a_id}) - '{role_a}' tai '{company_a}' ({row['cif_khdn_A']}) "
        f"co quan he [{relationship}] voi '{person_b_name}' ({person_b_id}) - '{role_b}' tai "
        f"'{company_b}' ({row['cif_khdn_B']})."
    )


def create_person_description(group):
    parts = []
    for _, row in group.iterrows():
        company = pick_display_name(row["name_khdn"], row["cif_khdn"])
        role = row["job_title"]
        high_role = row["high_role_group"]
        if pd.notna(high_role):
            parts.append(f"'{role}' tai {company} ({row['cif_khdn']}) [{high_role}]")
        else:
            parts.append(f"'{role}' tai {company} ({row['cif_khdn']})")
    return " | ".join(parts)


def build_grouped_by_person(relations_df, pairs_df):
    relation_pair_keys = set(
        relations_df.apply(
            lambda row: make_directed_tuple(row["cif_khdn_A"], row["cif_khdn_B"]),
            axis=1,
        )
    )

    pair_rows = pairs_df.copy()
    pair_rows["pair_tuple"] = pair_rows.apply(
        lambda row: make_directed_tuple(row["cif_khdn_A"], row["cif_khdn_B"]),
        axis=1,
    )
    pair_rows = pair_rows[pair_rows["pair_tuple"].isin(relation_pair_keys)].copy()

    links_a = pair_rows[
        ["cif_khcn", "name_khcn", "cif_khdn_A", "name_khdn_A", "job_title_A", "high_role_group_A"]
    ].rename(
        columns={
            "cif_khdn_A": "cif_khdn",
            "name_khdn_A": "name_khdn",
            "job_title_A": "job_title",
            "high_role_group_A": "high_role_group",
        }
    )
    links_b = pair_rows[
        ["cif_khcn", "name_khcn", "cif_khdn_B", "name_khdn_B", "job_title_B", "high_role_group_B"]
    ].rename(
        columns={
            "cif_khdn_B": "cif_khdn",
            "name_khdn_B": "name_khdn",
            "job_title_B": "job_title",
            "high_role_group_B": "high_role_group",
        }
    )

    all_links = pd.concat([links_a, links_b], ignore_index=True).drop_duplicates()

    grouped = (
        all_links.groupby(["cif_khcn", "name_khcn"], sort=False)
        .apply(
            lambda group: pd.Series(
                {
                    "danh_sach_doanh_nghiep": unique_list(group["name_khdn"]),
                    "so_luong_doanh_nghiep": group["cif_khdn"].nunique(),
                    "high_role_groups": unique_list(group["high_role_group"]),
                    "description": "Cac vai tro cua ca nhan: " + create_person_description(group),
                }
            )
        )
        .reset_index()
    )
    return grouped.sort_values(by=["so_luong_doanh_nghiep", "cif_khcn"], ascending=[False, True])


def write_excel_safe(path, pairs_df, people_df):
    try:
        with pd.ExcelWriter(path) as writer:
            pairs_df.to_excel(writer, index=False, sheet_name="pairs")
            people_df.to_excel(writer, index=False, sheet_name="by_person")
        print("Wrote Excel:", path.name)
    except PermissionError:
        print("Skipped Excel export because file is locked:", path.name)


def ensure_csv_input(csv_path, xlsx_path, label):
    if csv_path.exists():
        print(f"Using {label} CSV:", csv_path.name)
        return csv_path
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"Converted {label} from Excel to CSV:", xlsx_path.name, "->", csv_path.name)
        return csv_path
    raise FileNotFoundError(f"Cannot find {label}. Expected {csv_path.name} or {xlsx_path.name}.")


print("Helper functions ready.")


In [ ]:
start_time = time.time()

leader_input = ensure_csv_input(LEADER_FILE, LEADER_XLSX_FILE, "leader")
nclq_input = ensure_csv_input(NCLQ_FILE, NCLQ_XLSX_FILE, "nclq")

leader_header = pd.read_csv(leader_input, nrows=0)
leader_column_map = resolve_column_names(
    leader_header.columns,
    {
        "cif_khcn": ["cif_khcn"],
        "name_khcn": ["name_khcn"],
        "cif_khdn": ["cif_khdn"],
        "job_title": ["job_title"],
        "name_khdn": ["name_khdn"],
    },
    required_keys={"cif_khcn", "name_khcn", "cif_khdn", "job_title"},
)
leader_usecols = [col for col in leader_column_map.values() if col is not None]
df_leader = pd.read_csv(leader_input, usecols=leader_usecols).rename(
    columns={value: key for key, value in leader_column_map.items() if value is not None}
)
if "name_khdn" not in df_leader.columns:
    df_leader["name_khdn"] = df_leader["cif_khdn"]

df_leader = df_leader.dropna(subset=["cif_khcn", "cif_khdn"])
df_leader["cif_khcn"] = df_leader["cif_khcn"].astype(str).str.strip()
df_leader["name_khcn"] = df_leader["name_khcn"].astype(str).str.strip()
df_leader["cif_khdn"] = df_leader["cif_khdn"].astype(str).str.strip()
df_leader["name_khdn"] = df_leader["name_khdn"].astype(str).str.strip()
df_leader["job_title"] = df_leader["job_title"].astype(str).str.strip()

df_leader.loc[df_leader["name_khcn"].apply(normalize_text).isna(), "name_khcn"] = df_leader["cif_khcn"]
df_leader.loc[df_leader["name_khdn"].apply(normalize_text).isna(), "name_khdn"] = df_leader["cif_khdn"]

df_leader["job_title_normalized"] = df_leader["job_title"].apply(normalize_text)
df_leader = df_leader[df_leader["job_title_normalized"].notna()].copy()
df_leader["high_role_group"] = df_leader["job_title"].apply(classify_high_role)
df_leader["is_high_role"] = df_leader["high_role_group"].notna()

company_ids = set(df_leader["cif_khdn"].unique())
person_ids = set(df_leader["cif_khcn"].unique())
align_company_identifier = make_company_identifier_aligner(company_ids)
align_person_identifier = make_person_identifier_aligner(person_ids)

print("Leader rows:", len(df_leader))
print("Unique companies:", df_leader["cif_khdn"].nunique())
print("Unique people:", df_leader["cif_khcn"].nunique())
print("High-role rows:", int(df_leader["is_high_role"].sum()))
print("Load time:", round(time.time() - start_time, 2), "seconds")


In [ ]:
start_time = time.time()

merged = pd.merge(df_leader, df_leader, on="cif_khcn", suffixes=("_A", "_B"))
pairs_same_person = merged[merged["cif_khdn_A"] != merged["cif_khdn_B"]].copy()
pairs_same_person["name_khcn"] = pairs_same_person["name_khcn_A"]
pairs_same_person["link_source"] = "same_person"
pairs_same_person["family_relationship"] = pd.NA
pairs_same_person["person_pair_key"] = pd.NA
pairs_same_person["desc_line"] = pairs_same_person.apply(describe_same_person, axis=1)

nclq_raw = pd.read_csv(nclq_input)
nclq_column_map = resolve_column_names(
    nclq_raw.columns,
    {
        "CIF_NO": ["CIF_NO", "cif_no"],
        "CIF_NO_REL": ["CIF_NO_REL", "cif_no_rel"],
        "Mô tả MQH": ["Mô tả MQH", "Mo ta MQH", "M? t? MQH", "Mo ta moi quan he"],
        "Mã MQH": ["Mã MQH", "Ma MQH", "M? MQH"],
    },
    required_keys={"CIF_NO", "CIF_NO_REL"},
)
if not nclq_column_map.get("Mô tả MQH") and not nclq_column_map.get("Mã MQH"):
    raise KeyError("nclq.csv must contain either 'Mô tả MQH' or 'Mã MQH'.")
df_rel = nclq_raw.rename(
    columns={value: key for key, value in nclq_column_map.items() if value is not None}
)[[col for col in ["CIF_NO", "CIF_NO_REL", "Mô tả MQH", "Mã MQH"] if nclq_column_map.get(col)]].copy()
if "Mô tả MQH" not in df_rel.columns:
    df_rel["Mô tả MQH"] = pd.NA
df_rel["Mô tả MQH"] = df_rel["Mô tả MQH"].astype(str).str.strip()
df_rel["relationship_normalized"] = df_rel["Mô tả MQH"].apply(normalize_text)
if "Mã MQH" in df_rel.columns:
    df_rel["Mã MQH"] = df_rel["Mã MQH"].astype(str).str.strip()
    family_mask = df_rel["Mã MQH"].isin(FAMILY_RELATION_CODES)
else:
    family_mask = df_rel["relationship_normalized"].isin(FAMILY_RELATION_TYPES)
df_rel["relationship_display"] = df_rel["Mã MQH"].map(FAMILY_RELATION_CODE_LABELS) if "Mã MQH" in df_rel.columns else pd.NA
df_rel["relationship_display"] = df_rel["relationship_display"].fillna(df_rel["Mô tả MQH"])
df_family = df_rel[family_mask].copy()
df_family["CIF_NO"] = df_family["CIF_NO"].apply(align_person_identifier)
df_family["CIF_NO_REL"] = df_family["CIF_NO_REL"].apply(align_person_identifier)

merged_a = pd.merge(
    df_leader.add_suffix("_A"),
    df_family,
    left_on="cif_khcn_A",
    right_on="CIF_NO",
)
pairs_family = pd.merge(
    merged_a,
    df_leader.add_suffix("_B"),
    left_on="CIF_NO_REL",
    right_on="cif_khcn_B",
)
pairs_family = pairs_family[pairs_family["cif_khdn_A"] != pairs_family["cif_khdn_B"]].copy()
pairs_family = pairs_family[pairs_family["is_high_role_A"] & pairs_family["is_high_role_B"]].copy()
pairs_family["link_source"] = "family"
pairs_family["family_relationship"] = pairs_family["relationship_display"]
pairs_family["person_pair_key"] = pairs_family.apply(
    lambda row: make_undirected_tuple(row["cif_khcn_A"], row["cif_khcn_B"]),
    axis=1,
)
pairs_family["desc_line"] = pairs_family.apply(describe_family, axis=1)

family_row_a = pairs_family.copy()
family_row_a["cif_khcn"] = family_row_a["cif_khcn_A"]
family_row_a["name_khcn"] = family_row_a["name_khcn_A"]
family_row_b = pairs_family.copy()
family_row_b["cif_khcn"] = family_row_b["cif_khcn_B"]
family_row_b["name_khcn"] = family_row_b["name_khcn_B"]
pairs_family_split = pd.concat([family_row_a, family_row_b], ignore_index=True)

common_columns = [
    "cif_khdn_A",
    "name_khdn_A",
    "cif_khdn_B",
    "name_khdn_B",
    "cif_khcn",
    "name_khcn",
    "job_title_A",
    "job_title_B",
    "high_role_group_A",
    "high_role_group_B",
    "is_high_role_A",
    "is_high_role_B",
    "link_source",
    "family_relationship",
    "person_pair_key",
    "desc_line",
]

pairs = pd.concat(
    [pairs_same_person[common_columns], pairs_family_split[common_columns]],
    ignore_index=True,
)
pairs_all = pairs.copy()
pairs_hidden = pairs.copy()

print("same_person rows:", len(pairs_same_person))
print("family rows:", len(pairs_family_split))
print("combined pair rows:", len(pairs))
print("Build time:", round(time.time() - start_time, 2), "seconds")


In [ ]:
start_time = time.time()

df_declared = df_rel[["CIF_NO", "CIF_NO_REL"]].copy()
df_declared["left_raw"] = df_declared["CIF_NO"].apply(normalize_identifier)
df_declared["right_raw"] = df_declared["CIF_NO_REL"].apply(normalize_identifier)
df_declared["left_company"] = df_declared["left_raw"].apply(align_company_identifier)
df_declared["right_company"] = df_declared["right_raw"].apply(align_company_identifier)
df_declared["left_person"] = df_declared["left_raw"].apply(align_person_identifier)
df_declared["right_person"] = df_declared["right_raw"].apply(align_person_identifier)


def resolve_declared_type(row):
    left_types = []
    right_types = []

    if row["left_company"] in company_ids:
        left_types.append("company")
    if row["left_person"] in person_ids:
        left_types.append("person")
    if row["right_company"] in company_ids:
        right_types.append("company")
    if row["right_person"] in person_ids:
        right_types.append("person")

    if left_types == ["company"] and right_types == ["company"]:
        return "company_company"
    if left_types == ["person"] and right_types == ["person"]:
        return "person_person"
    if left_types == ["company"] and right_types == ["person"]:
        return "company_person"
    if left_types == ["person"] and right_types == ["company"]:
        return "person_company"
    return "ambiguous_or_unmatched"


df_declared["resolved_type"] = df_declared.apply(resolve_declared_type, axis=1)
df_declared[["CIF_NO", "CIF_NO_REL", "resolved_type"]].to_csv(
    DECLARED_DIAGNOSTIC_FILE,
    index=False,
    encoding="utf-8-sig",
)

declared_company_company = set(
    df_declared.loc[df_declared["resolved_type"] == "company_company"]
    .apply(lambda row: make_directed_tuple(row["left_company"], row["right_company"]), axis=1)
)
declared_person_person = set(
    df_declared.loc[df_declared["resolved_type"] == "person_person"]
    .apply(lambda row: make_undirected_tuple(row["left_person"], row["right_person"]), axis=1)
)

family_declared_mask = (
    pairs_hidden["link_source"].eq("family")
    & pairs_hidden["person_pair_key"].notna()
    & pairs_hidden["person_pair_key"].isin(declared_person_person)
)
if family_declared_mask.any():
    pairs_hidden = pairs_hidden.loc[~family_declared_mask].copy()

print("Declared company-company pairs:", len(declared_company_company))
print("Declared person-person pairs:", len(declared_person_person))
print("Removed declared family rows:", int(family_declared_mask.sum()))
print("Declared diagnostic file:", DECLARED_DIAGNOSTIC_FILE.name)
print("Declared resolution time:", round(time.time() - start_time, 2), "seconds")


In [ ]:
start_time = time.time()

for current_pairs in (pairs_all, pairs_hidden):
    current_pairs["high_role_in_A"] = current_pairs["job_title_A"].where(current_pairs["is_high_role_A"])
    current_pairs["high_role_in_B"] = current_pairs["job_title_B"].where(current_pairs["is_high_role_B"])
    current_pairs["high_group_in_A"] = current_pairs["high_role_group_A"].where(current_pairs["is_high_role_A"])
    current_pairs["high_group_in_B"] = current_pairs["high_role_group_B"].where(current_pairs["is_high_role_B"])

all_relations = (
    pairs_all.groupby(
        ["cif_khdn_A", "name_khdn_A", "cif_khdn_B", "name_khdn_B"],
        sort=False,
    )
    .agg(
        shared_individuals=("cif_khcn", unique_list),
        shared_individual_names=("name_khcn", unique_list),
        roles_in_A=("job_title_A", unique_list),
        roles_in_B=("job_title_B", unique_list),
        high_roles_in_A=("high_role_in_A", unique_list),
        high_roles_in_B=("high_role_in_B", unique_list),
        high_role_groups_in_A=("high_group_in_A", unique_list),
        high_role_groups_in_B=("high_group_in_B", unique_list),
        connection_types=("link_source", unique_list),
        family_relationships=("family_relationship", unique_list),
        description=("desc_line", lambda values: " | ".join(unique_list(values))),
    )
    .reset_index()
)
all_relations["shared_count"] = all_relations["shared_individuals"].apply(len)
all_relations = all_relations.sort_values(
    by=["shared_count", "cif_khdn_A", "cif_khdn_B"],
    ascending=[False, True, True],
)

pair_columns = [
    "cif_khdn_A",
    "name_khdn_A",
    "cif_khdn_B",
    "name_khdn_B",
    "shared_count",
    "shared_individuals",
    "shared_individual_names",
    "roles_in_A",
    "roles_in_B",
    "high_roles_in_A",
    "high_roles_in_B",
    "high_role_groups_in_A",
    "high_role_groups_in_B",
    "connection_types",
    "family_relationships",
    "description",
]
all_relations = all_relations[pair_columns]
all_people = build_grouped_by_person(all_relations, pairs_all)

hidden_relations = (
    pairs_hidden.groupby(
        ["cif_khdn_A", "name_khdn_A", "cif_khdn_B", "name_khdn_B"],
        sort=False,
    )
    .agg(
        shared_individuals=("cif_khcn", unique_list),
        shared_individual_names=("name_khcn", unique_list),
        roles_in_A=("job_title_A", unique_list),
        roles_in_B=("job_title_B", unique_list),
        high_roles_in_A=("high_role_in_A", unique_list),
        high_roles_in_B=("high_role_in_B", unique_list),
        high_role_groups_in_A=("high_group_in_A", unique_list),
        high_role_groups_in_B=("high_group_in_B", unique_list),
        connection_types=("link_source", unique_list),
        family_relationships=("family_relationship", unique_list),
        description=("desc_line", lambda values: " | ".join(unique_list(values))),
    )
    .reset_index()
)
hidden_relations["shared_count"] = hidden_relations["shared_individuals"].apply(len)
hidden_relations = hidden_relations.sort_values(
    by=["shared_count", "cif_khdn_A", "cif_khdn_B"],
    ascending=[False, True, True],
)
hidden_relations = hidden_relations[pair_columns]
hidden_relations = hidden_relations[
    ~hidden_relations.apply(
        lambda row: make_directed_tuple(row["cif_khdn_A"], row["cif_khdn_B"]) in declared_company_company,
        axis=1,
    )
].copy()
hidden_people = build_grouped_by_person(hidden_relations, pairs_hidden)

all_relations.to_csv(ALL_PAIRS_FILE, index=False, encoding="utf-8-sig")
all_people.to_csv(ALL_PEOPLE_FILE, index=False, encoding="utf-8-sig")
write_excel_safe(ALL_XLSX_FILE, all_relations, all_people)

hidden_relations.to_csv(HIDDEN_PAIRS_FILE, index=False, encoding="utf-8-sig")
hidden_people.to_csv(HIDDEN_PEOPLE_FILE, index=False, encoding="utf-8-sig")
write_excel_safe(HIDDEN_XLSX_FILE, hidden_relations, hidden_people)

print("All pairs:", len(all_relations))
print("All people:", len(all_people))
print("Hidden pairs:", len(hidden_relations))
print("Hidden people:", len(hidden_people))
print("Saved:", ALL_PAIRS_FILE.name, ALL_PEOPLE_FILE.name, ALL_XLSX_FILE.name)
print("Saved:", HIDDEN_PAIRS_FILE.name, HIDDEN_PEOPLE_FILE.name, HIDDEN_XLSX_FILE.name)
print("Saved:", DECLARED_DIAGNOSTIC_FILE.name)
print("Export time:", round(time.time() - start_time, 2), "seconds")
